# Notebook 3 — Counterfactual Resolution Simulator

**Mandate Vacuum Governance Intelligence**

This notebook simulates alternate mandate ownership structures and evaluates how they would affect resolution efficiency compared to the current baseline.

### Approach
1. Measure baseline Entropy + Half-Life per category
2. Simulate: reassign primary ownership to each possible department
3. Compare: projected entropy, half-life, and resolution improvement
4. Recommend: best alternate structure per category

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import entropy as shannon_entropy
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
df = pd.read_csv('../sample_data/bbmp_complaints_cleaned.csv')
print(f'Loaded {len(df)} transfer records across {df["complaint_id"].nunique()} complaints.')
df.head()

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def calc_entropy(probs):
    """Normalized Shannon entropy from a probability series."""
    H = shannon_entropy(probs, base=2)
    n = len(probs)
    H_max = np.log2(n) if n > 1 else 1.0
    return round(H / H_max, 4) if H_max > 0 else 0.0

def calc_halflife(n_transfers, total_days, scaling=0.5):
    """Compute accountability half-life in days."""
    if total_days == 0:
        return float('inf')
    lam = (n_transfers / total_days) * scaling
    return round(np.log(2) / lam, 1) if lam > 0 else float('inf')

def baseline_metrics(df, category):
    """Compute real entropy and average half-life for a category."""
    cat = df[df['category'] == category]
    probs = cat['to_dept'].value_counts() / len(cat)
    H = calc_entropy(probs)
    
    # Avg half-life across complaints in this category
    hl_list = []
    for cid in cat['complaint_id'].unique():
        c = cat[cat['complaint_id'] == cid]
        hl = calc_halflife(len(c), c['days_open'].max())
        if hl != float('inf'):
            hl_list.append(hl)
    avg_hl = round(np.mean(hl_list), 1) if hl_list else 0.0
    
    res_rate = cat.groupby('complaint_id')['resolved'].last().mean()
    avg_days = cat.groupby('complaint_id')['days_open'].max().mean()
    
    return {
        'entropy': H,
        'avg_half_life': avg_hl,
        'resolution_rate': round(res_rate, 3),
        'avg_days_open': round(avg_days, 1)
    }

print('Helper functions defined.')

In [ ]:
def simulate_scenario(df, category, primary_dept, primary_share=0.80):
    """
    Simulate what happens if primary_dept takes primary_share of transfers.
    
    Parameters:
        df           : Transfer records
        category     : Complaint category
        primary_dept : Department assigned as primary owner
        primary_share: Fraction of transfers handled by primary dept (default 0.80)
    
    Returns:
        dict with simulated metrics and improvement deltas
    """
    cat = df[df['category'] == category]
    all_depts = list(set(cat['to_dept'].unique()) | set(cat['from_dept'].unique()))
    other_depts = [d for d in all_depts if d != primary_dept]
    
    # Simulated distribution
    if not other_depts:
        sim_probs = pd.Series({primary_dept: 1.0})
    else:
        remaining = 1.0 - primary_share
        other_share = remaining / len(other_depts)
        sim_dist = {primary_dept: primary_share}
        for d in other_depts:
            sim_dist[d] = other_share
        sim_probs = pd.Series(sim_dist)
    
    sim_entropy = calc_entropy(sim_probs)
    
    # Fewer transfers → longer half-life (primary owner resolves faster)
    # Estimate: primary ownership reduces average transfers by (1 - primary_share)
    baseline = baseline_metrics(df, category)
    reduction_factor = 1 - (primary_share - 0.5)  # 0.5 baseline → scales down transfers
    avg_transfers = cat.groupby('complaint_id').size().mean()
    avg_days = baseline['avg_days_open']
    sim_transfers = avg_transfers * reduction_factor
    sim_hl = calc_halflife(sim_transfers, avg_days)
    
    # Projected resolution improvement (heuristic: lower entropy + longer HL → better resolution)
    entropy_improvement = baseline['entropy'] - sim_entropy
    hl_improvement = (sim_hl - baseline['avg_half_life']) / max(baseline['avg_half_life'], 1)
    projected_res_delta = round((entropy_improvement * 0.4 + hl_improvement * 0.3) * 100, 1)
    sim_resolution_rate = min(1.0, baseline['resolution_rate'] + projected_res_delta / 100)
    
    return {
        'category': category,
        'primary_dept': primary_dept,
        'sim_entropy': sim_entropy,
        'sim_half_life': sim_hl,
        'sim_resolution_rate': round(sim_resolution_rate, 3),
        'delta_entropy': round(baseline['entropy'] - sim_entropy, 4),
        'delta_half_life': round(sim_hl - baseline['avg_half_life'], 1),
        'delta_resolution_pct': projected_res_delta,
        'baseline_entropy': baseline['entropy'],
        'baseline_half_life': baseline['avg_half_life'],
        'baseline_resolution': baseline['resolution_rate']
    }

print('Simulator function defined.')

In [ ]:
# Run simulations for all categories × all departments
all_depts = list(set(df['to_dept'].unique()) | set(df['from_dept'].unique()))
categories = df['category'].unique()

all_scenarios = []
for cat in categories:
    for dept in all_depts:
        scenario = simulate_scenario(df, cat, dept)
        all_scenarios.append(scenario)

sim_df = pd.DataFrame(all_scenarios)

# Best scenario per category
best_scenarios = sim_df.loc[sim_df.groupby('category')['delta_resolution_pct'].idxmax()]

print('=== COUNTERFACTUAL SIMULATION RESULTS ===')
print('Best mandate reassignment per category:\n')
display_cols = ['category', 'primary_dept', 'baseline_entropy', 'sim_entropy',
                'baseline_half_life', 'sim_half_life', 'baseline_resolution',
                'sim_resolution_rate', 'delta_resolution_pct']
print(best_scenarios[display_cols].to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cat_list = list(best_scenarios['category'])
x = np.arange(len(cat_list))
width = 0.35

# Chart 1: Entropy — Baseline vs Best Scenario
ax1 = axes[0]
b1 = ax1.bar(x - width/2, best_scenarios['baseline_entropy'], width,
             label='Baseline (Current)', color='#e74c3c', alpha=0.8)
b2 = ax1.bar(x + width/2, best_scenarios['sim_entropy'], width,
             label='Best Scenario (Simulated)', color='#27ae60', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(cat_list)
ax1.set_ylabel('Normalized Ownership Entropy')
ax1.set_title('Entropy: Baseline vs\nBest Mandate Reassignment', fontsize=13, fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 1.1)
for bar in b1:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{bar.get_height():.2f}', ha='center', fontsize=8)
for bar in b2:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{bar.get_height():.2f}', ha='center', fontsize=8)

# Chart 2: Projected resolution improvement
ax2 = axes[1]
colors = ['#27ae60' if v > 0 else '#e74c3c' for v in best_scenarios['delta_resolution_pct']]
bars = ax2.bar(cat_list, best_scenarios['delta_resolution_pct'], color=colors, alpha=0.85, edgecolor='white')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_ylabel('Projected Resolution Rate Improvement (%)')
ax2.set_title('Resolution Efficiency Gain\nfrom Best Mandate Reassignment', fontsize=13, fontweight='bold')
for bar, val in zip(bars, best_scenarios['delta_resolution_pct']):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + (0.5 if val >= 0 else -1.5),
             f'+{val:.1f}%' if val >= 0 else f'{val:.1f}%',
             ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/03_counterfactual_simulator.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/03_counterfactual_simulator.png')

In [ ]:
# Final policy recommendations
print('=== POLICY RECOMMENDATIONS ===')
print('Based on counterfactual simulation:\n')
for _, row in best_scenarios.iterrows():
    delta = row['delta_resolution_pct']
    emoji = '🔼' if delta > 0 else '➡️'
    print(f"{emoji} {row['category']:12s}")
    print(f"   Assign PRIMARY ownership to: {row['primary_dept']}")
    print(f"   Entropy:     {row['baseline_entropy']:.2f} → {row['sim_entropy']:.2f}  (↓ {row['delta_entropy']:.2f})")
    print(f"   Half-Life:   {row['baseline_half_life']:.1f}d → {row['sim_half_life']:.1f}d  (↑ {row['delta_half_life']:.1f}d)")
    print(f"   Projected resolution improvement: +{delta:.1f}%\n")